**NYC Taxi Trips - Basics**

**Dataset**: samples.nyctaxi.trips

**Difficulty**: Easy

**Topics:** filter, count, aggregation, sorting

In [0]:
from pyspark.sql import functions as f, types as t

Learn — Loading, Filtering, and Aggregating
- Function	What it does
- spark.table("catalog.schema.table")	Loads a Delta/Unity Catalog table into a DataFrame
- df.count()	Triggers compute and returns the total number of rows (action)
- df.filter(condition)	Returns rows matching the condition (transformation — lazy)
- df.groupBy(col).agg(...)	Groups rows by a column and applies aggregate functions
- df.orderBy(col.desc())	Sorts rows in descending order
- F.col("name")	References a column by name
- F.avg(), F.min(), F.max(), F.count()	Aggregation functions

In [0]:
df=spark.table("samples.nyctaxi.trips")

print("total rows", df.count())
#filter to long trips (over 5 miles)
filtered_df=df.filter(f.col("trip_distance")>5)

#compute trip stats by pickup ZIP code
df.groupBy("pickup_zip").agg(f.count("*").alias("total_trips"), f.round(f.avg("fare_amount"),2).alias("avg_fare")).orderBy(f.col("total_trips").desc()).show()

**Problem 1**

Count the total number of trips in the NYC taxi dataset. Load the table samples.nyctaxi.trips and return a single-row DataFrame that tells us how many records exist.

Expected output columns:

total_trips (bigint) - total number of trip records

In [0]:
df.printSchema()


In [0]:
df=spark.read.table("samples.nyctaxi.trips")

result_1=df.agg(f.count("*").alias("total_trips"))
result_1.show()

In [0]:

# ── Tests for Problem 1 ──────────────────────────────────────────
assert result_1 is not None, "result_1 is None - did you forget to assign your DataFrame?"
assert hasattr(result_1, 'columns'), "result_1 must be a Spark DataFrame"
cols = [c.lower() for c in result_1.columns]
assert 'total_trips' in cols, "Missing column: total_trips"
assert len(cols) == 1, f"Expected exactly 1 columns, got {len(cols)}: {cols}"
cnt = result_1.count()
assert cnt == 1, f"Expected exactly 1 row, got {cnt}"
total = result_1.collect()[0]['total_trips']
assert total > 0, f"Expected total_trips > 0, got {total}"
assert total < 10_000_000, f"total_trips seems unreasonably large: {total}"
print(f"Problem 1 passed ✓  ({cnt} rows, total_trips={total})")